<a href="https://colab.research.google.com/github/jampoStyle/SPD-AI-Assistant/blob/main/Latest_v5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import os
import google.generativeai as genai
from dotenv import load_dotenv # Import the function to load variables from .env
import textwrap # Used for formatting text
from chromadb import PersistentClient # For the vector database
# Note: SentenceTransformer is used by ChromaDB's default embedding function,
# so we don't need to import it directly here unless using a custom one.
import glob # To find files matching a pattern
from typing import List
import streamlit as st # Import the Streamlit library
import re # Import regular expressions for more robust string cleaning
# Import libraries for reading other document types
import PyPDF2
import docx
import json
from PIL import Image
import pytesseract

# --- Load Environment Variables EARLY ---
# Load environment variables from .env file as early as possible.
# This looks for a file named .env in the same directory and loads key=value pairs
load_dotenv()

# --- Configuration ---
# Define the name of your default local CSV file
CSV_FILE_PATH = 'missing_instruments.csv'

# Define the directory where your knowledge documents are stored
KNOWLEDGE_DOCS_DIR = 'knowledge_docs'

# Define the path for the ChromaDB persistent storage
CHROMA_DB_PATH = 'chroma_db'

# --- Streamlit UI Setup ---
st.set_page_config(page_title="SPAississtant: Your SPD AI Assistant", layout="wide")
st.title("SPAississtant: Your SPD AI Assistant")

# --- Knowledge Base Loading and Processing (for RAG) ---
# Use Streamlit's cache decorator to run this function only once
@st.cache_resource
def setup_rag_knowledge_base(docs_dir: str, db_path: str):
    """Sets up the RAG knowledge base by loading, chunking, and embedding documents."""
    print("Setting up RAG knowledge base...") # This will print to the terminal running Streamlit
    rag_client = None
    rag_collection = None
    knowledge_documents = []
    document_chunks = []

    if os.path.exists(docs_dir):
        knowledge_documents = load_documents(docs_dir) # Call the updated load_documents
        if knowledge_documents:
            document_chunks = chunk_documents(knowledge_documents)
            if document_chunks:
                # Create or load the vector store
                rag_client, rag_collection = create_vector_store(document_chunks, db_path)
                print("RAG setup complete.")
            else:
                 print("No document chunks created for RAG.")
        else:
            print(f"No supported documents found in {docs_dir} for RAG.")
    else:
        st.warning(f"Knowledge documents directory not found at '{docs_dir}'. RAG will not be available.")
        print(f"Knowledge documents directory not found at {docs_dir}. RAG will not be available.")


    return rag_client, rag_collection # Return both client and collection

# Updated load_documents function to handle .txt, .pdf, and .docx
def load_documents(directory: str) -> List[str]:
    """Loads text content from .txt, .pdf, .docx, and .png files in a directory."""
    documents = []
    supported_files = []

    # Find all supported files
    supported_files.extend(glob.glob(os.path.join(directory, '*.txt')))
    supported_files.extend(glob.glob(os.path.join(directory, '*.pdf')))
    supported_files.extend(glob.glob(os.path.join(directory, '*.docx')))
    supported_files.extend(glob.glob(os.path.join(directory, '*.png')))

    if not supported_files:
        st.warning(f"No supported files (.txt, .pdf, .docx, .png) found in '{directory}' for the knowledge base. RAG will not be available.")
        return []

    st.info(f"Loading documents from '{directory}'...")
    st.info(f"Found {len(supported_files)} supported files: {[os.path.basename(f) for f in supported_files]}")

    for file_path in supported_files:
        try:
            file_extension = os.path.splitext(file_path)[1].lower()
            content = ""
            file_name = os.path.basename(file_path)
            st.info(f"Processing file: {file_name}")

            if file_extension == '.txt':
                with open(file_path, 'r', encoding='utf-8') as f:
                    content = f.read()
            elif file_extension == '.pdf':
                with open(file_path, 'rb') as f:
                    reader = PyPDF2.PdfReader(f)
                    if reader.is_encrypted:
                        try:
                            reader.decrypt('')
                        except Exception:
                            st.warning(f"Could not decrypt PDF: {file_path}")
                            continue
                    for page in reader.pages:
                        content += page.extract_text() or ""
            elif file_extension == '.docx':
                try:
                    doc = docx.Document(file_path)
                    # Extract text from paragraphs
                    content = "\n".join([para.text for para in doc.paragraphs])
                    # Extract text from tables
                    for table in doc.tables:
                        for row in table.rows:
                            for cell in row.cells:
                                content += "\n" + cell.text
                    st.success(f"Successfully loaded DOCX file: {file_name}")
                except Exception as e:
                    st.error(f"Error processing DOCX file {file_path}: {str(e)}")
                    continue
            elif file_extension == '.png':
                try:
                    image = Image.open(file_path)
                    content = pytesseract.image_to_string(image)
                except Exception as e:
                    st.warning(f"Could not process PNG file {file_path}: {e}")
                    continue

            # Clean up content and add file information
            content = content.strip()
            if content:
                # Add file information at the start of the content
                content = f"File: {file_name}\n\n{content}"
                documents.append(content)
                st.success(f"Successfully processed {file_name}")
            else:
                st.warning(f"No extractable text found in {file_path}.")
        except Exception as e:
            st.error(f"Error reading {file_path}: {str(e)}")

    st.info(f"Successfully loaded {len(documents)} documents")
    return documents

def chunk_documents(documents: List[str], chunk_size: int = 500, chunk_overlap: int = 50) -> List[str]:
    """Breaks down documents into smaller chunks."""
    chunks = []
    if not documents:
        return []

    # st.info(f"Chunking documents (size: {chunk_size}, overlap: {chunk_overlap})...") # Optional: show chunking progress
    for doc in documents:
        # Replace multiple newlines with single ones to clean up spacing from extraction
        cleaned_doc = re.sub(r'\n\s*\n', '\n', doc).strip()
        words = cleaned_doc.split()
        for i in range(0, len(words), chunk_size - chunk_overlap):
            chunk = " ".join(words[i:i + chunk_size])
            chunks.append(chunk)
    st.info(f"Created {len(chunks)} chunks.")
    return chunks

def create_vector_store(chunks: List[str], db_path: str):
    """Creates or loads a ChromaDB vector store and adds document chunks."""
    client = PersistentClient(path=db_path)
    collection_name = "spd_knowledge_base"

    try:
        collection = client.get_collection(name=collection_name)

        if chunks:
            existing_ids = set()
            try:
                existing_ids = set(collection.get(include=[])['ids'])
            except Exception as e:
                print(f"Could not retrieve existing ChromaDB IDs: {e}")
                st.warning("Could not verify existing documents in ChromaDB. May add duplicates.")

            chunks_to_add = [chunk for i, chunk in enumerate(chunks) if f"chunk_{i}" not in existing_ids]

            if chunks_to_add:
                st.info(f"Adding {len(chunks_to_add)} new chunks to the collection.")
                ids_to_add = [f"chunk_{i + len(existing_ids)}" for i in range(len(chunks_to_add))]
                # Add metadata for each chunk with file information
                metadatas = []
                for i, chunk in enumerate(chunks_to_add):
                    # Extract file name from the chunk text
                    file_name = "Unknown Source"
                    if "File:" in chunk:
                        file_name = chunk.split("File:")[1].split("\n")[0].strip()
                    metadatas.append({
                        "source": file_name,
                        "chunk_id": ids_to_add[i],
                        "index": i
                    })

                collection.add(
                    documents=chunks_to_add,
                    ids=ids_to_add,
                    metadatas=metadatas
                )
                st.success(f"Added {len(chunks_to_add)} chunks to the collection. Total chunks: {collection.count()}")
            else:
                st.info("No new chunks to add. Collection is up to date.")

    except Exception as e:
        st.info(f"Creating new collection: '{collection_name}'")
        print(f"Error getting collection (creating new one): {e}")
        collection = client.create_collection(name=collection_name)
        if chunks:
            st.info(f"Adding {len(chunks)} chunks to the new collection...")
            ids = [f"chunk_{i}" for i in range(len(chunks))]
            # Add metadata for each chunk with file information
            metadatas = []
            for i, chunk in enumerate(chunks):
                # Extract file name from the chunk text
                file_name = "Unknown Source"
                if "File:" in chunk:
                    file_name = chunk.split("File:")[1].split("\n")[0].strip()
                metadatas.append({
                    "source": file_name,
                    "chunk_id": ids[i],
                    "index": i
                })

            collection.add(
                documents=chunks,
                ids=ids,
                metadatas=metadatas
            )
            st.success(f"Added {len(chunks)} chunks to the new collection.")
        else:
            st.warning("No chunks to add to the new collection.")

    return client, collection

def find_relevant_chunks(query: str, collection, n_results: int = 5) -> List[dict]:
    """Searches the vector store for chunks relevant to the query."""
    if collection is None or collection.count() == 0:
        st.warning("Knowledge base is empty. Cannot retrieve relevant information.")
        return []

    try:
        results = collection.query(
            query_texts=[query],
            n_results=n_results,
            include=['documents', 'metadatas', 'distances']
        )

        relevant_chunks = []
        if results and results['documents'] and results['documents'][0]:
            for doc, metadata, distance in zip(
                results['documents'][0],
                results['metadatas'][0],
                results['distances'][0]
            ):
                # Extract source from metadata or document content
                source = "Unknown Source"
                if metadata and 'source' in metadata:
                    source = metadata['source']
                elif "File:" in doc:
                    source = doc.split("File:")[1].split("\n")[0].strip()

                relevant_chunks.append({
                    'text': doc,
                    'source': source,
                    'chunk_id': metadata.get('chunk_id', 'Unknown ID') if metadata else 'Unknown ID',
                    'relevance': 1 - (distance / 2)  # Convert distance to relevance score
                })

        return relevant_chunks
    except Exception as e:
        st.error(f"Error searching knowledge base: {str(e)}")
        return []

# --- Data Processing for AI Prompt ---
# This function is updated to handle optional file upload and local file loading,
# with improved CSV reading and column normalization diagnostics.
def prepare_data_for_ai(uploaded_file, default_local_path):
    """Loads data from an uploaded or default local CSV file and extracts key information for the AI prompt."""
    df = None
    source_description = "No data loaded."

    if uploaded_file is not None:
        try:
            # Try reading with default pandas parameters first
            df = pd.read_csv(uploaded_file)
            st.success("CSV loaded successfully from uploaded file.")
            source_description = f"Data loaded from uploaded file: {uploaded_file.name}"
        except pd.errors.EmptyDataError:
            st.error("Error: Uploaded CSV file is empty.")
            source_description = "Uploaded CSV file is empty."
        except Exception as e:
            # If default read fails, try with a common alternative encoding
            try:
                uploaded_file.seek(0) # Reset file pointer to the beginning
                df = pd.read_csv(uploaded_file, encoding='latin1')
                st.success("CSV loaded successfully from uploaded file (using latin1 encoding).")
                source_description = f"Data loaded from uploaded file: {uploaded_file.name} (latin1 encoding)"
            except Exception as e2:
                 st.error(f"Error loading uploaded CSV with default or latin1 encoding: {e2}")
                 source_description = f"Error loading uploaded CSV with default or latin1 encoding: {e2}"

    else:
        # If no file uploaded, try to load from the default local path
        st.info(f"No file uploaded. Attempting to load default CSV from '{default_local_path}'...")
        if os.path.exists(default_local_path):
            try:
                # Try reading with default pandas parameters first
                df = pd.read_csv(default_local_path)
                st.success(f"CSV loaded successfully from default local file: '{default_local_path}'")
                source_description = f"Data loaded from default local file: {default_local_path}"
            except pd.errors.EmptyDataError:
                st.error(f"Error: Default local CSV file '{default_local_path}' is empty.")
                source_description = f"Default local CSV file '{default_local_path}' is empty."
            except Exception as e:
                 # If default read fails, try with a common alternative encoding
                 try:
                     df = pd.read_csv(default_local_path, encoding='latin1')
                     st.success(f"CSV loaded successfully from default local file (using latin1 encoding): '{default_local_path}'")
                     source_description = f"Data loaded from default local file: {default_local_path} (latin1 encoding)"
                 except Exception as e2:
                     st.error(f"Error loading default local CSV with default or latin1 encoding: {e2}")
                     source_description = f"Error loading default local CSV with default or latin1 encoding: {e2}"

        else:
            st.warning(f"Default local CSV file not found at '{default_local_path}'. Please upload a file.")
            source_description = f"Default local CSV file not found at '{default_local_path}'."


    if df is not None and not df.empty:
        st.write("Raw Columns Read:", df.columns.tolist()) # Show raw columns

        # Normalize column names: strip whitespace and convert to lowercase for robust matching
        # Corrected approach: Iterate and clean each column name string
        cleaned_columns = []
        for col in df.columns:
            # Remove any character that is NOT a letter, number, or underscore, and strip whitespace
            cleaned_col = re.sub(r'[^a-z0-9_]+', '', col.strip().lower())
            cleaned_columns.append(cleaned_col)
        df.columns = cleaned_columns # Assign the list of cleaned names back
        st.write("Normalized Columns:", df.columns.tolist()) # Show normalized columns

        # --- Monthly Analysis ---
        monthly_summary_string = "Monthly Missing Instruments Summary: Not available ('datefirstmissing' column missing or invalid format)"
        if 'datefirstmissing' in df.columns:
            try:
                # Convert 'datefirstmissing' to datetime objects
                df['datefirstmissing'] = pd.to_datetime(df['datefirstmissing'])
                # Extract month and year
                df['Month_Year'] = df['datefirstmissing'].dt.to_period('M')
                # Count missing instruments per month
                monthly_counts = df['Month_Year'].value_counts().sort_index()

                if not monthly_counts.empty:
                    monthly_summary_string = "\nMonthly Missing Instruments Summary:\n"
                    for period, count in monthly_counts.items():
                        monthly_summary_string += f"- {period.strftime('%Y-%m')}: {count}\n"
                else:
                     monthly_summary_string = "\nMonthly Missing Instruments Summary: No data with valid dates."

            except Exception as e:
                monthly_summary_string = f"\nMonthly Missing Instruments Summary: Error processing date column - {e}"
                st.warning(monthly_summary_string) # Show warning in UI if date processing fails

        # --- Technician Analysis ---
        technician_summary_string = "Technician Missing Instruments Summary: Not available ('spdtechnician' column missing)"
        if 'spdtechnician' in df.columns:
            # Count missing instruments per technician
            technician_counts = df['spdtechnician'].value_counts()

            if not technician_counts.empty:
                technician_summary_string = "\nTechnician Missing Instruments Summary:\n"
                for technician, count in technician_counts.items():
                    technician_summary_string += f"- {technician}: {count}\n"
            else:
                 technician_summary_string = "\nTechnician Missing Instruments Summary: No data in 'spdtechnician' column."

        # --- Most Common Missing Instrument Analysis ---
        most_common_instrument_string = "Most Common Missing Instrument: Not available (requires 'prod' or 'instrumentname' column)"
        # Use normalized column names for checking
        prod_col_norm = 'prod'
        inst_name_col_norm = 'instrumentname'

        # Find the actual column name that corresponds to 'Prod #' or 'Instrument_Name' after normalization
        instrument_column = None
        if prod_col_norm in df.columns:
            instrument_column = prod_col_norm
        elif inst_name_col_norm in df.columns:
            instrument_column = inst_name_col_norm

        if instrument_column:
            # Count occurrences of each instrument in the identified column
            instrument_counts = df[instrument_column].value_counts()
            if not instrument_counts.empty:
                most_common_instrument = instrument_counts.index[0]
                most_common_count = instrument_counts.iloc[0]
                most_common_instrument_string = f"Most Common Missing Instrument (by {instrument_column}): {most_common_instrument} (Missing {most_common_count} times)"
            else:
                most_common_instrument_string = f"Most Common Missing Instrument: No data in '{instrument_column}' column."
        else:
            most_common_instrument_string = "Most Common Missing Instrument: Requires 'prod' or 'instrumentname' column."
            st.info(most_common_instrument_string) # Show message in UI about missing columns for this analysis


        # --- Pattern Identification (Technician, Tray, Instrument, Date) ---
        pattern_analysis_string = "Detailed Pattern Analysis: Not available (requires 'trayname', 'prod' or 'instrumentname', 'datefirstmissing', 'spdtechnician' columns)"
        # Use normalized column names for checking
        required_cols_for_patterns = ['trayname', 'datefirstmissing', 'spdtechnician']
        # Add the identified instrument column to the required list if it exists
        if instrument_column:
            required_cols_for_patterns.append(instrument_column)

        # Check if all required columns for pattern analysis are present
        if all(col in df.columns for col in required_cols_for_patterns):
             try:
                 pattern_analysis_string = "\nDetailed Pattern Analysis:\n"

                 # Top Technician + Tray combinations
                 tech_tray_patterns = df.groupby(['spdtechnician', 'trayname']).size().sort_values(ascending=False)
                 if not tech_tray_patterns.empty:
                     pattern_analysis_string += "\nTop Technician + Tray Combinations with Missing Instruments:\n"
                     # Limit to top N combinations for brevity in prompt
                     for (tech, tray), count in tech_tray_patterns.head(10).items():
                         pattern_analysis_string += f"- Technician: {tech}, Tray: {tray}, Count: {count}\n"

                 # Top Technician + Instrument combinations (using the identified instrument column)
                 if instrument_column:
                     tech_instrument_patterns = df.groupby(['spdtechnician', instrument_column]).size().sort_values(ascending=False)
                     if not tech_instrument_patterns.empty:
                         pattern_analysis_string += f"\nTop Technician + {instrument_column} Combinations with Missing Instruments:\n"
                          # Limit to top N combinations for brevity in prompt
                         for (tech, item), count in tech_instrument_patterns.head(10).items():
                             pattern_analysis_string += f"- Technician: {tech}, {instrument_column}: {item}, Count: {count}\n"
                 else:
                      pattern_analysis_string += "\nTop Technician + Instrument Combinations: Instrument column ('prod' or 'instrumentname') not found.\n"


                 # Top Technicians by Date/Month (already covered by monthly + technician summaries, but can add cross-ref)
                 # The AI can correlate the existing summaries.

             except Exception as e:
                 pattern_analysis_string = f"\nDetailed Pattern Analysis: Error during pattern processing - {e}"
                 st.warning(pattern_analysis_string) # Show warning in UI if pattern processing fails
        else:
            # Show which specific normalized columns are missing
            missing_cols = [col for col in required_cols_for_patterns if col not in df.columns]
            pattern_analysis_string = f"\nDetailed Pattern Analysis: Requires columns: {', '.join(missing_cols)}"
            st.info(pattern_analysis_string) # Show message in UI about missing columns


        # --- General Summary ---
        total_missing_entries = len(df)
        # Use normalized column names for checking
        unique_instruments = df[prod_col_norm].nunique() if prod_col_norm in df.columns else 'N/A'
        unique_manufacturers = df['manufacturer'].nunique() if 'manufacturer' in df.columns else 'N/A'

        # Get a list of missing instruments and their quantities (simplified)
        # Use normalized column names here
        prod_col = 'prod' if 'prod' in df.columns else None
        qty_col = 'quantity missing' if 'quantity missing' in df.columns else None
        inst_name_col = 'instrument name' if 'instrument name' in df.columns else None

        if prod_col and qty_col:
            missing_summary = df[[prod_col, qty_col]].to_string(index=False, header=True)
        elif inst_name_col and qty_col:
             missing_summary = df[[inst_name_col, qty_col]].to_string(index=False, header=True)
        else:
            missing_summary = df.head().to_string(index=False, header=True)


        # Include comments if available
        comments = ""
        if 'missing comments' in df.columns:
            comments_col = 'missing comments' if 'missing comments' in df.columns else None
            valid_comments = df[comments_col].dropna().tolist()
            if valid_comments:
                comments = "\n\nRelevant Missing Comments:\n" + "\n".join(valid_comments)

        # Construct the final data summary string including all analyses
        data_summary_string = f"""
        Overall Missing Instruments Summary:
        Total entries in report: {total_missing_entries}
        Unique Product IDs: {unique_instruments}
        Unique Manufacturers: {unique_manufacturers}

        {most_common_instrument_string} # <-- Include the most common instrument

        {monthly_summary_string}

        {technician_summary_string}

        {pattern_analysis_string} # <-- Include the new pattern analysis

        Detailed Missing Items (Sample/Partial List):
        {missing_summary}
        {comments}
        """
        return data_summary_string

    else:
        # If df is None or empty after trying both upload and local file
        st.warning("No valid data loaded from uploaded file or default local file.")
        return "No valid data loaded."


# --- AI Setup ---
# Load environment variables from .env file.
# Moved load_dotenv() to the top of the script

# Get your Gemini API key from the environment (loaded from .env or system environment)
# This happens outside the cached function but after load_dotenv()
api_key = os.getenv("GEMINI_API_KEY")

# Configure the Gemini API (runs once)
@st.cache_resource
def configure_gemini_api(): # Removed parameter here
    """Configures the Gemini API."""
    # Retrieve the API key INSIDE the cached function
    api_key_value = os.getenv("GEMINI_API_KEY")
    print(f"Attempting to configure Gemini API with key: {api_key_value[:5] if api_key_value else 'None'}...") # Diagnostic print

    if api_key_value:
        try:
            # Explicitly pass the api_key_value to the configure function
            genai.configure(api_key=api_key_value)
            st.success("Gemini API configured.")
            print("Gemini API configured successfully.") # Diagnostic print
            # Select the model
            model = genai.GenerativeModel('gemini-1.5-flash-latest') # Select model inside function
            print(f"AI Model selected: {model.model_name}") # Diagnostic print
            return model # Return the model object

        except Exception as e:
            st.error(f"Error configuring Gemini API: {e}")
            st.error("Please check if your API key is correct and valid in the .env file.")
            print(f"Error during genai.configure or model selection: {e}") # Diagnostic print
            return None # Return None if configuration fails
    else:
        st.warning("GEMINI_API_KEY environment variable not set. AI features will not be available.")
        st.warning("Please ensure the .env file exists and contains your API key.")
        print("API key not found in environment.") # Diagnostic print
        return None # Return None if API key is missing

def display_sources(sources):
    """
    Display a Sources/References section on the Streamlit page.
    Args:
        sources (list): List of source file names or dicts with 'name' and 'snippet'.
    """
    st.markdown("### Sources / References")
    if sources:
        for src in sources:
            if isinstance(src, dict):
                st.markdown(f"""
                **Source**: {src.get('name', 'Unknown Source')}
                **Relevance**: {src.get('relevance', 'N/A')}
                **Content**: {src.get('snippet', '')}
                """)
            else:
                st.markdown(f"- {src}")
    else:
        st.info("No sources or references found for this answer.")

def display_analysis_sections(analysis: str):
    """
    Display the comprehensive analysis in well-formatted sections.
    """
    st.markdown("## Comprehensive Analysis")

    # Split the analysis into sections
    sections = analysis.split("\n\n")

    for section in sections:
        if section.strip():
            # Check for section headers
            if section.startswith("#"):
                st.markdown(section)
            # Check for bullet points
            elif section.startswith("-") or section.startswith("*"):
                st.markdown(section)
            # Check for numbered lists
            elif re.match(r'^\d+\.', section):
                st.markdown(section)
            # Regular text
            else:
                st.write(section)

def analyze_data_with_context(query: str, data_summary: str, relevant_chunks: List[dict]) -> str:
    """
    Perform comprehensive analysis combining data and knowledge base context.
    """
    # Prepare the context from relevant chunks
    knowledge_context = "\n\n".join([chunk['text'] for chunk in relevant_chunks])

    # Create the analysis prompt
    prompt = f"""You are an AI assistant for a Sterile Processing Department (SPD) consultant.
    Your goal is to provide clear, actionable recommendations based on the provided data and knowledge.
    Analyze the following summary of missing instruments in the context of the user's query.

    Adopt the perspectives of:
    - A healthcare executive
    - An SPD manager
    - A continuous improvement master
    - An SPD technician
    - Someone with business acumen in SPD

    Perform a detailed analysis of the provided Missing Instruments Data, looking for patterns and correlations between:
    - Technicians and the frequency/types of missing instruments.
    - Dates/Months and the frequency of missing instruments.
    - If 'Tray Name' and 'Prod #' columns are available, analyze patterns between Technicians, Trays, and specific Instruments.

    Based on your analysis of the data patterns AND the provided SPD knowledge, provide recommendations focusing on:
    1. Potential root causes and triggers for these specific patterns
    2. Operational or process improvements in SPD
    3. Business or inventory management considerations
    4. Continuous improvement strategies

    Format the output using the following structure:
    # Executive Summary
    [Provide a brief overview of the key findings and recommendations]

    # Multiple Perspectives Analysis
    ## Healthcare Executive View
    [Analysis from executive perspective]

    ## SPD Manager View
    [Analysis from manager perspective]

    ## Continuous Improvement View
    [Analysis from improvement perspective]

    ## SPD Technician View
    [Analysis from technician perspective]

    ## Business Acumen View
    [Analysis from business perspective]

    # Pattern Analysis
    ## Technician Patterns
    [Analysis of technician-related patterns]

    ## Temporal Patterns
    [Analysis of date/month patterns]

    ## Instrument/Tray Patterns
    [Analysis of instrument and tray patterns]

    # Detailed Recommendations
    ## Root Causes
    [List of identified root causes]

    ## Operational Improvements
    [List of operational improvements]

    ## Business Considerations
    [List of business considerations]

    ## Continuous Improvement Strategies
    [List of improvement strategies]

    # Prioritized Action Items
    1. [Highest priority item]
    2. [Second priority item]
    3. [Third priority item]
    [etc.]

    When referencing the provided SPD knowledge, refer to the concepts or practices discussed, not the document names.

    If the data or context is ambiguous or insufficient, state clearly what additional information is needed.

    User Query: {query}

    Missing Instruments Data Summary:
    {data_summary}

    SPD Knowledge Context:
    {knowledge_context}

    Analysis and Recommendations:"""

    # Get response from AI model
    response = AI_MODEL.generate_content(prompt)
    return response.text

def rag_query(query: str):
    """
    Process a user query using RAG and return both answer and sources.
    Args:
        query (str): The user's question
    Returns:
        tuple: (answer, sources, analysis)
    """
    if not rag_collection or not AI_MODEL:
        return "RAG system is not properly configured. Please check your knowledge base and AI model setup.", [], None

    try:
        # Get relevant chunks from the knowledge base
        relevant_chunks = find_relevant_chunks(query, rag_collection)

        if not relevant_chunks:
            return "I couldn't find any relevant information in the knowledge base to answer your question.", [], None

        # Prepare the context from relevant chunks
        context = "\n\n".join([chunk['text'] for chunk in relevant_chunks])

        # Create the prompt for the AI model
        prompt = f"""Based on the following context, please answer the question.
        If the answer cannot be found in the context, say so.
        Also, please cite the specific sources you used to form your answer.

        Context:
        {context}

        Question: {query}

        Answer:"""

        # Get response from AI model
        response = AI_MODEL.generate_content(prompt)
        answer = response.text

        # Prepare sources using the metadata
        sources = [
            {
                "name": chunk['source'],
                "snippet": chunk['text'][:200] + "..." if len(chunk['text']) > 200 else chunk['text'],
                "relevance": f"{chunk['relevance']:.2%}"
            }
            for chunk in relevant_chunks
        ]

        return answer, sources, relevant_chunks

    except Exception as e:
        st.error(f"Error processing query: {str(e)}")
        return "Sorry, I encountered an error while processing your question.", [], None

# --- Main Streamlit Application Flow ---

# Setup the RAG knowledge base (runs once)
rag_client, rag_collection = setup_rag_knowledge_base(KNOWLEDGE_DOCS_DIR, CHROMA_DB_PATH)

# Configure the Gemini API (runs once)
AI_MODEL = configure_gemini_api()

# File Uploader for CSV (Now Optional)
uploaded_file = st.file_uploader("Upload your Missing Instrument Summary CSV (Optional)", type=['csv'])

# Process CSV and prepare data for AI (handles uploaded or local file)
data_summary_string = prepare_data_for_ai(uploaded_file, CSV_FILE_PATH)

# Query Box
user_query = st.text_input("Ask a question:")

if user_query:
    try:
        # Get RAG-based answer and sources
        answer, sources, relevant_chunks = rag_query(user_query)

        # Display the answer in a clear section
        st.markdown("## Knowledge Base Answer")
        st.markdown(answer)

        # Display sources with detailed information
        st.markdown("## Sources and References")
        display_sources(sources)

        # If we have CSV data, perform and display comprehensive analysis
        if data_summary_string and data_summary_string != "No valid data loaded.":
            analysis = analyze_data_with_context(user_query, data_summary_string, relevant_chunks)
            display_analysis_sections(analysis)

    except Exception as e:
        st.error(f"Error processing your question: {str(e)}")

# --- Instructions/Notes for the user ---
st.sidebar.subheader("How to Use")
st.sidebar.info(
    """
    1. Ensure your `.env` file in the project folder contains `GEMINI_API_KEY="YOUR_API_KEY"`.
    2. Place your SPD knowledge documents (.txt, .pdf, .docx) in the `knowledge_docs` folder.
    3. Upload your Missing Instrument Summary CSV file (optional).
    4. Enter your question or request in the query box.
    5. View the AI-generated answer, sources, and comprehensive analysis (if CSV is loaded).
    """
)
st.sidebar.subheader("Project Status")
st.sidebar.text(f"Knowledge Docs Folder: {KNOWLEDGE_DOCS_DIR}")
st.sidebar.text(f"Chroma DB Path: {CHROMA_DB_PATH}")

if rag_collection:
    st.sidebar.text(f"Knowledge Base Chunks: {rag_collection.count()}")
else:
    st.sidebar.text("Knowledge Base: Not loaded")

if AI_MODEL:
     st.sidebar.text(f"AI Model: {AI_MODEL.model_name}")
else:
    st.sidebar.text("AI Model: Not configured")

def main():
    # ...main code...
    KNOWLEDGE_BASE = load_documents(KNOWLEDGE_DOCS_DIR)
    # ...rest of main...

if __name__ == '__main__':
    main()
